In [2]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import heapq as pq
import pandas as pd


In [8]:
def pad(inp,size=9):
    return '0'*(size-len(str(inp)))+str(inp)

def suffix_trim(x, trim_length, size=9):
    return x[:len(x)-trim_length]

def prefix_trim(x, trim_length, size=9):
    return x[trim_length:len(x)]

In [9]:
delta=0.01
n=50

def generate_samples(n,q=4,sample_size=1000):
    sample_inp=[]
    for i in range(sample_size):
        inp=np.random.randint(0,q,n)
        sample_inp.append(''.join(map(str,inp)))
        
    return sample_inp

sample_inp=generate_samples(n,4,1000)

In [6]:
def channel1(trace, N,n,q=4):
    trim_lengths=np.random.randint(0,n,N)
    traces=[trace]*N
    traces=list(map(suffix_trim,traces,trim_lengths))
    
    for i in range(N):
        suffix_append_len=trim_lengths[i]
        suffix_append=np.random.randint(0,q,suffix_append_len)
        traces[i]=traces[i]+''.join(map(str, suffix_append))
    
    return traces

def algo(trace,N,n,q=4):
    traces=channel1(trace, N,n,q)
    estimate=''
    num_count=defaultdict(int)
    traces_list=defaultdict(list)
    for i in range(n):
        maxnum=0
        maxc=0
        if len(traces)==0:
            estimate+=str(0)
        else:
            for j in range(len(traces)):
                try:
                    num_count[int(traces[j][i])] += 1
                    traces_list[int(traces[j][i])].append(traces[j])
                    if num_count[int(traces[j][i])] > maxc:
                        maxc=num_count[int(traces[j][i])]
                        maxnum=int(traces[j][i])
                except IndexError:
                    pass
                    
            estimate+=str(maxnum)
            traces=traces_list[maxnum]
    
    answer=0
    if estimate==trace:
        answer+=1

    return answer  

In [ ]:
def channel2(trace, N, n, q=4):
    trim_lengths_suffix=np.random.randint(0,n//2,N)
    trim_lengths_prefix=np.random.randint(0,n//2,N)
    
    traces=[trace]*N
    traces=list(map(suffix_trim,traces,trim_lengths_suffix))
    traces=list(map(prefix_trim,traces,trim_lengths_prefix))
    
    for i in range(N):
        suffix_append_len=trim_lengths_suffix[i]
        prefix_append_len=trim_lengths_prefix[i]
        suffix_append=np.random.randint(0,q,suffix_append_len)
        traces[i]=traces[i]+''.join(map(str, suffix_append))
        
        prefix_append=np.random.randint(0,q,prefix_append_len)
        traces[i]=''.join(map(str, prefix_append))+traces[i]
        
        
        # print(traces[i])
    
    return traces
def algo2_channel2(trace, N, n, q=4):
    """Majority voting algorithm starting from MIDDLE for channel2 traces.
    Expands outward (left and right) from the middle position.
    Uses alphabet size q (default 4: digits 0-3)."""
    traces = channel2(trace, N, n, q)
    
    # Find average trace length (traces may have different lengths due to variable trimming)
    avg_len = np.mean([len(t) for t in traces])
    mid_pos = int(avg_len / 2)
    
    estimate = ['?'] * n  # Initialize with unknowns
    active_indices = set(range(len(traces)))  # Track which traces are still valid
    
    # Expand from middle outward: first process middle, then alternate left/right
    processed = set()
    
    # Start from approximate middle and expand outward
    for offset in range(n):
        positions_to_process = []
        
        # Alternate: middle, middle-1, middle+1, middle-2, middle+2, ...
        if offset == 0:
            positions_to_process = [mid_pos]
        else:
            if mid_pos - offset >= 0:
                positions_to_process.append(mid_pos - offset)
            if mid_pos + offset < n:
                positions_to_process.append(mid_pos + offset)
        
        for pos in positions_to_process:
            if pos in processed or pos < 0 or pos >= n:
                continue
            
            processed.add(pos)
            
            # Collect digits at this position from active traces
            digits_at_pos = []
            for idx in active_indices:
                if pos < len(traces[idx]):
                    try:
                        digits_at_pos.append(int(traces[idx][pos]))
                    except (ValueError, IndexError):
                        pass
            
            if not digits_at_pos:
                # No valid traces, can't decide
                estimate[pos] = '0'
                continue
            
            # Majority voting
            digit_counts = defaultdict(int)
            for digit in digits_at_pos:
                digit_counts[digit] += 1
            
            majority_digit = max(digit_counts, key=digit_counts.get)
            estimate[pos] = str(majority_digit)
            
            # Remove traces that don't match the majority vote at this position
            traces_to_remove = []
            for idx in list(active_indices):
                if pos < len(traces[idx]):
                    try:
                        if int(traces[idx][pos]) != majority_digit:
                            traces_to_remove.append(idx)
                    except (ValueError, IndexError):
                        traces_to_remove.append(idx)
            
            for idx in traces_to_remove:
                active_indices.discard(idx)
            
            # Stop if no active traces remain
            if not active_indices:
                break
        
        if not active_indices:
            break
    
    # Check if estimate matches the original trace
    estimate_str = ''.join(estimate)
    success = 1 if estimate_str == trace else 0
    
    return success

def channel3(trace, N, n, q=4):
    trim_lengths_suffix=np.random.randint(0,n,N)
    
    traces=[trace]*N
    traces=list(map(suffix_trim,traces,trim_lengths_suffix,[n]*N))
    
    for i in range(N):
        suffix_append_len=np.random.randint(0,n,N)
        suffix_append=np.random.randint(0,q,suffix_append_len)
        traces[i]=traces[i]+''.join(map(str, suffix_append))

    return traces

SyntaxError: unterminated triple-quoted string literal (detected at line 122) (4247299172.py, line 109)

In [ ]:
def success_rate(inputs, N, n):
    success=0
    for inp in inputs:
        success+=algo(inp, N, n)
    return success/len(inputs)

def success_rate_channel2(inputs, N, n):
    success = 0
    for inp in inputs:
        success += algo2_channel2(inp, N, n)
    return success / len(inputs)

## Trying to speed up PFM using LLMs

In [ ]:
def algo_fast(trace, N, n):
    """
    Fast majority voting algorithm using NumPy vectorization and bitwise operations.
    Only keeps trace strings that match the majority vote at each position.
    Same logic as algo() but optimized for speed. Alphabet size 4 (digits 0-3).
    """
    traces = channel1(trace, N, n)
    
    # Convert traces to NumPy array for fast vectorized operations
    try:
        traces_arr = np.array([np.array(list(t), dtype=np.uint8) for t in traces], dtype=np.uint8)
    except:
        # Fallback for irregular trace lengths
        traces_arr = None
    
    estimate = []
    
    if traces_arr is not None and traces_arr.shape[0] > 0:
        # Use bitwise majority voting with active mask
        active_mask = np.ones(len(traces), dtype=bool)
        
        for i in range(n):
            if not np.any(active_mask):
                estimate.append(0)
                continue
            
            # Get digits at position i for active traces only
            digits_at_i = traces_arr[active_mask, i]
            
            # Find majority digit using bincount (fast bitwise operation)
            digit_counts = np.bincount(digits_at_i, minlength=4)
            majority_digit = np.argmax(digit_counts)
            
            estimate.append(majority_digit)
            
            # Update active mask: keep only traces matching majority digit at position i
            # This uses bitwise AND to filter efficiently
            active_mask &= (traces_arr[:, i] == majority_digit)
    else:
        # Fallback to original algorithm for edge cases
        num_count = defaultdict(int)
        traces_list = defaultdict(list)
        for i in range(n):
            maxnum = 0
            maxc = 0
            if len(traces) == 0:
                estimate.append(0)
            else:
                for j in range(len(traces)):
                    try:
                        num_count[int(traces[j][i])] += 1
                        traces_list[int(traces[j][i])].append(traces[j])
                        if num_count[int(traces[j][i])] > maxc:
                            maxc = num_count[int(traces[j][i])]
                            maxnum = int(traces[j][i])
                    except IndexError:
                        pass
                
                estimate.append(maxnum)
                traces = traces_list[maxnum]
                num_count.clear()
                traces_list.clear()
    
    estimate_str = ''.join(map(str, estimate))
    return 1 if estimate_str == trace else 0

In [ ]:
# ==================================================================================
# SWEEP: N vs n for fixed success rate = 0.99
# ==================================================================================

import time

def sweep_N_vs_n_fast(target_success_rate=0.99, n_min=50, n_max=1000, n_step=50, num_trials=100):
    """
    Sweep N vs n using fast optimized algo_fast function.
    Finds minimum N for each n that achieves target success rate.
    Uses binary search with exponential upper bound search.
    
    Args:
        target_success_rate: Target success rate (default 0.99)
        n_min, n_max, n_step: Range and step for n values
        num_trials: Number of random trials per N test
    
    Returns:
        dict: {n: min_N} mapping
        list: n_values used
    """
    results = {}
    n_values = list(range(n_min, n_max + 1, n_step))
    
    print(f"Sweeping N vs n (target success rate: {target_success_rate})")
    print(f"n range: {n_min} to {n_max}, step: {n_step}")
    print(f"Trials per N: {num_trials}")
    print("=" * 80)
    
    prev_N = None
    
    for n_idx, n in enumerate(n_values, 1):
        print(f"\n[{n_idx}/{len(n_values)}] n={n}: ", flush=True)
        
        # PHASE 1: Exponential search for upper bound
        print(f"  Phase 1 - Finding upper bound...", flush=True)
        if prev_N is not None:
            # Start from previous N (monotonically increasing)
            N_high = max(prev_N, 10)
        else:
            # First iteration: start with reasonable estimate
            N_high = max(10, n // 5)
        
        # Keep doubling until we find upper bound
        max_iter_phase1 = 0
        while max_iter_phase1 < 20:
            print(f" N={N_high}", end="", flush=True)
            
            # Test with random samples (alphabet size 4: digits 0-3)
            # Use FIXED SEED for reproducible results
            rng = np.random.RandomState(42)
            test_inputs = [''.join(map(str, rng.randint(0, 4, n))) for _ in range(num_trials)]
            success = sum(algo_fast(inp, N_high, n) for inp in test_inputs)
            sr = success / num_trials
            
            print(f"({sr:.0%})", end="", flush=True)
            
            if sr >= target_success_rate:
                break
            N_high = int(N_high * 1.5) + 1
            max_iter_phase1 += 1
        
        N_low = max(1, int(N_high * 0.5))
        
        # PHASE 2: Binary search for exact minimum N
        print(f"\n  Phase 2 - Binary search [{N_low}, {N_high}]...", flush=True)
        
        max_iter_phase2 = 0
        while N_high - N_low > 1 and max_iter_phase2 < 20:
            N_mid = (N_low + N_high) // 2
            print(f" N={N_mid}", end="", flush=True)
            
            # Use FIXED SEED for reproducible results
            rng = np.random.RandomState(42)
            test_inputs = [''.join(map(str, rng.randint(0, 4, n))) for _ in range(num_trials)]
            success = sum(algo_fast(inp, N_mid, n) for inp in test_inputs)
            sr = success / num_trials
            
            print(f"({sr:.0%})", end="", flush=True)
            
            if sr >= target_success_rate:
                N_high = N_mid
            else:
                N_low = N_mid
            max_iter_phase2 += 1
        
        min_N = N_high
        print(f"\n           ✓ Found minimum N={min_N}")
        results[n] = min_N
        prev_N = min_N
    
    n_array = np.array(n_values)
    N_array = np.array([results[n] for n in n_values])
    
    for i in range(1, len(n_values)):
        if results[n_values[i]] < results[n_values[i-1]]:
            old_N = results[n_values[i]]
            results[n_values[i]] = results[n_values[i-1]]
    
    print("\nSUMMARY - N vs n Results:")
    print("-" * 80)
    for n in n_values:
        print(f"  n={n:>4d}  →  N={results[n]:>6d}")
    
    # Save results to CSV file
    import csv
    filename = 'sweep_N_vs_n_fast.csv'
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'N_min'])
        for n in n_values:
            writer.writerow([n, results[n]])
    print(f"\n✓ Results saved to '{filename}'")
    
    return results, n_values

print("Starting N vs n sweep with optimized algo_fast...")
start_time = time.time()
sweep_results_fast, n_values_sweep = sweep_N_vs_n_fast(target_success_rate=0.99, 
                                                        n_min=50, n_max=1000, n_step=50, 
                                                        num_trials=100)
elapsed = time.time() - start_time
print(f"\n✓ Sweep complete in {elapsed:.1f}s")


In [ ]:

# ==================================================================================
# PLOT: N vs n sweep results
# ==================================================================================

import matplotlib.pyplot as plt
import numpy as np

print("Plotting sweep results...")

if 'sweep_results_fast' not in globals() or not sweep_results_fast:
    print("Error: sweep_results_fast not found. Please run the sweep cell first.")
else:
    # Extract data
    n_vals = np.array(n_values_sweep)
    N_vals = np.array([sweep_results_fast[n] for n in n_values_sweep])
    
    # Create figure with single plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot: Linear scale
    ax.plot(n_vals, N_vals, 'o-', linewidth=3, markersize=10, color='#1f77b4', label='Min N for 99%')
    ax.fill_between(n_vals, N_vals * 0.98, N_vals * 1.02, alpha=0.15, color='#1f77b4')
    ax.set_xlabel('n (sequence length)', fontsize=13, fontweight='bold')
    ax.set_ylabel('N (minimum traces needed)', fontsize=13, fontweight='bold')
    ax.set_title('Minimum Traces vs Sequence Length (Success Rate = 99%)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=12, loc='upper left')
    
    # Add value labels
    for n, N in zip(n_vals, N_vals):
        ax.annotate(f'{N}', xy=(n, N), xytext=(0, 8), 
                    textcoords='offset points', ha='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('sweep_N_vs_n_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print analysis
    print("\n" + "="*80)
    print("SWEEP RESULTS ANALYSIS:")
    print("="*80)
    print(f"Data points: {len(n_vals)}")
    print(f"N range: {N_vals.min()} - {N_vals.max()}")
    print(f"n range: {n_vals.min()} - {n_vals.max()}")
    
    if len(n_vals) > 1:
        # Calculate growth rate
        growth_rate = (N_vals[-1] / N_vals[0]) ** (1 / (n_vals[-1] - n_vals[0]))
        print(f"\nGrowth rate: {growth_rate:.4f}x per unit increase in n")
        
        # N/n ratio analysis
        ratios = N_vals / n_vals
        print(f"N/n ratio: {ratios[0]:.4f} (at n={n_vals[0]}) → {ratios[-1]:.4f} (at n={n_vals[-1]})")
        
        if ratios[-1] > ratios[0]:
            print("→ SUPERLINEAR growth (N/n increasing)")
        else:
            print("→ SUBLINEAR growth (N/n decreasing)")
    
    print("="*80)
    print("✓ Plot saved as 'sweep_N_vs_n_results.png'")


Channel 2 and 3:

In [ ]:
# ==================================================================================
# OPTIMIZED FAST VERSIONS FOR CHANNEL2 AND CHANNEL3
# ==================================================================================

def algo_fast_channel2(trace, N, n, q=4):
    """
    Fast majority voting for channel2 (left-to-right with active mask filtering).
    Same logic as algo2_channel2 but using NumPy vectorization.
    Alphabet size q (default 4: digits 0-3).
    """
    traces = channel2(trace, N, n, q)
    
    # Convert traces to NumPy array for fast vectorized operations
    try:
        # Pad to equal length
        max_len = max(len(t) for t in traces) if traces else n
        traces_arr = np.zeros((len(traces), max_len), dtype=np.uint8)
        for i, t in enumerate(traces):
            traces_arr[i, :len(t)] = np.array(list(t), dtype=np.uint8)
    except:
        traces_arr = None
    
    estimate = []
    
    if traces_arr is not None and traces_arr.shape[0] > 0:
        # Use majority voting with active mask (left-to-right)
        active_mask = np.ones(len(traces), dtype=bool)
        
        for i in range(n):
            if not np.any(active_mask):
                estimate.append(0)
                continue
            
            # Get digits at position i for active traces only
            digits_at_i = traces_arr[active_mask, i]
            
            # Find majority digit using bincount
            digit_counts = np.bincount(digits_at_i[digits_at_i > 0], minlength=q)
            if len(digit_counts) == 0 or np.all(digit_counts == 0):
                estimate.append(0)
                active_mask.fill(False)
                continue
            
            majority_digit = np.argmax(digit_counts)
            estimate.append(majority_digit)
            
            # Update active mask: keep only traces matching majority digit at position i
            active_mask &= (traces_arr[:, i] == majority_digit)
    else:
        # Fallback
        estimate = [0] * n
    
    estimate_str = ''.join(map(str, estimate))
    return 1 if estimate_str == trace else 0


def algo_fast_channel3(trace, N, n, q=4):
    """
    Fast majority voting for channel3 (left-to-right with active mask filtering).
    Uses NumPy vectorization with alphabet size q (default 4: digits 0-3).
    """
    traces = channel3(trace, N, n, q)
    
    # Convert traces to NumPy array for fast vectorized operations
    try:
        traces_arr = np.array([np.array(list(t), dtype=np.uint8) for t in traces], dtype=np.uint8)
    except:
        traces_arr = None
    
    estimate = []
    
    if traces_arr is not None and traces_arr.shape[0] > 0:
        # Use majority voting with active mask (left-to-right)
        active_mask = np.ones(len(traces), dtype=bool)
        
        for i in range(n):
            if not np.any(active_mask):
                estimate.append(0)
                continue
            
            # Get digits at position i for active traces only
            digits_at_i = traces_arr[active_mask, i]
            
            # Find majority digit using bincount
            digit_counts = np.bincount(digits_at_i, minlength=q)
            majority_digit = np.argmax(digit_counts)
            
            estimate.append(majority_digit)
            
            # Update active mask: keep only traces matching majority digit at position i
            active_mask &= (traces_arr[:, i] == majority_digit)
    else:
        estimate = [0] * n
    
    estimate_str = ''.join(map(str, estimate))
    return 1 if estimate_str == trace else 0


In [ ]:

# ==================================================================================
# SWEEP FUNCTIONS FOR CHANNEL2 AND CHANNEL3
# ==================================================================================

import time

def sweep_N_vs_n_channel2(target_success_rate=0.99, n_min=50, n_max=1000, n_step=50, num_trials=100, q=4):
    """
    Sweep N vs n using fast optimized algo_fast_channel2 function.
    Finds minimum N for each n that achieves target success rate.
    Channel2: both suffix and prefix trimming. Alphabet size q (default 4).
    """
    results = {}
    n_values = list(range(n_min, n_max + 1, n_step))
    
    print(f"Sweeping N vs n - CHANNEL2 (target success rate: {target_success_rate}, q={q})")
    print(f"n range: {n_min} to {n_max}, step: {n_step}")
    print(f"Trials per N: {num_trials}")
    print("=" * 80)
    
    prev_N = None
    
    for n_idx, n in enumerate(n_values, 1):
        print(f"\n[{n_idx}/{len(n_values)}] n={n}: ", flush=True)
        
        # PHASE 1: Exponential search for upper bound
        print(f"  Phase 1 - Finding upper bound...", flush=True)
        if prev_N is not None:
            N_high = prev_N
        else:
            N_high = max(10, n // 10)
        
        while True:
            print(f" N={N_high}", end="", flush=True)
            test_inputs = [''.join(map(str, np.random.randint(0, q, n))) for _ in range(num_trials)]
            success = sum(algo_fast_channel2(inp, N_high, n, q) for inp in test_inputs)
            sr = success / num_trials
            print(f"({sr:.0%})", end="", flush=True)
            
            if sr >= target_success_rate:
                break
            N_high = int(N_high * 2)
        
        N_low = max(1, N_high // 2)
        
        # PHASE 2: Binary search for exact minimum N
        print(f"\n  Phase 2 - Binary search [{N_low}, {N_high}]...", flush=True)
        
        while N_high - N_low > 1:
            N_mid = (N_low + N_high) // 2
            print(f" N={N_mid}", end="", flush=True)
            
            test_inputs = [''.join(map(str, np.random.randint(0, q, n))) for _ in range(num_trials)]
            success = sum(algo_fast_channel2(inp, N_mid, n, q) for inp in test_inputs)
            sr = success / num_trials
            print(f"({sr:.0%})", end="", flush=True)
            
            if sr >= target_success_rate:
                N_high = N_mid
            else:
                N_low = N_mid
        
        min_N = N_high
        print(f"\n           ✓ Found minimum N={min_N}")
        results[n] = min_N
        prev_N = min_N
    
    print("\n" + "=" * 80)
    print("SUMMARY - Channel2 N vs n Results:")
    print("-" * 80)
    for n in n_values:
        print(f"  n={n:>4d}  →  N={results[n]:>6d}")
    
    # Save results to CSV file
    import csv
    filename = 'sweep_N_vs_n_channel2.csv'
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'N_min'])
        for n in n_values:
            writer.writerow([n, results[n]])
    print(f"✓ Results saved to '{filename}'")
    
    return results


def sweep_N_vs_n_channel3(target_success_rate=0.99, n_min=50, n_max=1000, n_step=50, num_trials=100, q=4):
    """
    Sweep N vs n using fast optimized algo_fast_channel3 function.
    Finds minimum N for each n that achieves target success rate.
    Channel3: suffix trimming only. Alphabet size q (default 4).
    """
    results = {}
    n_values = list(range(n_min, n_max + 1, n_step))
    
    print(f"Sweeping N vs n - CHANNEL3 (target success rate: {target_success_rate}, q={q})")
    print(f"n range: {n_min} to {n_max}, step: {n_step}")
    print(f"Trials per N: {num_trials}")
    print("=" * 80)
    
    prev_N = None
    
    for n_idx, n in enumerate(n_values, 1):
        print(f"\n[{n_idx}/{len(n_values)}] n={n}: ", flush=True)
        
        # PHASE 1: Exponential search for upper bound
        print(f"  Phase 1 - Finding upper bound...", flush=True)
        if prev_N is not None:
            N_high = prev_N
        else:
            N_high = max(10, n // 10)
        
        while True:
            print(f" N={N_high}", end="", flush=True)
            test_inputs = [''.join(map(str, np.random.randint(0, q, n))) for _ in range(num_trials)]
            success = sum(algo_fast_channel3(inp, N_high, n, q) for inp in test_inputs)
            sr = success / num_trials
            print(f"({sr:.0%})", end="", flush=True)
            
            if sr >= target_success_rate:
                break
            N_high = int(N_high * 2)
        
        N_low = max(1, N_high // 2)
        
        # PHASE 2: Binary search for exact minimum N
        print(f"\n  Phase 2 - Binary search [{N_low}, {N_high}]...", flush=True)
        
        while N_high - N_low > 1:
            N_mid = (N_low + N_high) // 2
            print(f" N={N_mid}", end="", flush=True)
            
            test_inputs = [''.join(map(str, np.random.randint(0, q, n))) for _ in range(num_trials)]
            success = sum(algo_fast_channel3(inp, N_mid, n, q) for inp in test_inputs)
            sr = success / num_trials
            print(f"({sr:.0%})", end="", flush=True)
            
            if sr >= target_success_rate:
                N_high = N_mid
            else:
                N_low = N_mid
        
        min_N = N_high
        print(f"\n           ✓ Found minimum N={min_N}")
        results[n] = min_N
        prev_N = min_N
    
    print("\n" + "=" * 80)
    print("SUMMARY - Channel3 N vs n Results:")
    print("-" * 80)
    for n in n_values:
        print(f"  n={n:>4d}  →  N={results[n]:>6d}")
    
    # Save results to CSV file
    import csv
    filename = 'sweep_N_vs_n_channel3.csv'
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'N_min'])
        for n in n_values:
            writer.writerow([n, results[n]])
    print(f"✓ Results saved to '{filename}'")
    
    return results

print("✓ Optimized channel2 and channel3 functions loaded")


## Maximum Likelihood

In [ ]:
from itertools import product
import time

def maximum_likelihood(trace, N, n, q=2, verbose=False):
    traces = channel1(trace, N, n, q)
    
    # Function to calculate common prefix length between two strings
    def common_prefix_length(s1, s2):
        length = 0
        for i in range(min(len(s1), len(s2))):
            if s1[i] == s2[i]:
                length += 1
            else:
                break
        return length
    
    # Find the candidate string with maximum likelihood
    max_likelihood = -1
    best_candidate = None
    total_candidates = q ** n
    
    if verbose:
        print(f"Searching through {total_candidates} candidate strings (q={q}, n={n})")
    
    # Iterate through all strings in q^n
    for idx, digits in enumerate(product(range(q), repeat=n)):
        if verbose and (idx % 100 == 0):
            print(f"  Progress: {idx}/{total_candidates}")
        
        candidate = ''.join(map(str, digits))
        
        # Calculate likelihood as sum of common prefix lengths
        likelihood = sum(common_prefix_length(candidate, trace) for trace in traces)
        
        if likelihood > max_likelihood:
            max_likelihood = likelihood
            best_candidate = candidate
    
    # Check if the best candidate matches the original trace
    success = 1 if best_candidate == trace else 0
    
    if verbose:
        print(f"Best candidate: {best_candidate}, Original trace: {trace}, Success: {success}")
    
    return success


def success_rate_ml(N,n,q):
    sample_inp=generate_samples(n,q,1000)
    # print(sample_inp)
    success=0
    count=0
    for inp in sample_inp:
        print(count)
        success+=maximum_likelihood(inp, N, n, q)
        count+=1
        print(f"Current success rate: {success/count:.4f} after {count} samples")
    return success/1000


def sweep_N_vs_n_ml(target_success_rate=0.99, q=2, num_samples=1000):
    """
    Sweep N vs n using VANILLA maximum_likelihood (brute force approach).
    Uses exponential search + binary search to find EXACT minimum N.
    OPTIMIZED: Uses previous n results to estimate starting N for next n.
    n varies from 10 to 20 in intervals of 2.
    Tests with 1000 samples for robust success rate estimation.
    WARNING: This is computationally expensive but will give definitive results!
    """
    results = {}
    n_values = list(range(10, 21, 2)) 
    
    print(f"Sweeping N vs n using VANILLA maximum_likelihood")
    print(f"Target success rate: {target_success_rate}")
    print(f"Samples per test: {num_samples}")
    print("=" * 75)
    
    prev_N = None
    prev_n = None
    prev_growth = None
    
    for n in n_values:
        print(f"\nn={n}: (q={q}, checking all {q}^{n}={q**n} candidates per test)")
        
        # PHASE 1: Exponential search to find upper bound
        print(f"  Phase 1 - Finding upper bound...")
        
        # Smart initialization: use previous results to estimate starting point
        if prev_N is not None and prev_n is not None:
            delta_n = n - prev_n
            # Use observed growth rate (account for convergence)
            if prev_growth is not None:
                # Use moving average of growth to smooth out noise
                estimated_growth = (prev_growth + (prev_N / prev_prev_N if prev_prev_N else 1.0)) / 2
            else:
                # First step: estimate conservatively
                estimated_growth = (prev_N / prev_prev_N) if prev_prev_N else 1.2
            estimated_N = int(prev_N * (estimated_growth ** delta_n))
            N_high = max(estimated_N, prev_N)  # Don't go below previous N (should be monotonic)
            prev_growth = estimated_N / prev_N
            prev_prev_N = prev_N
        else:
            # First iteration: start with reasonable estimate
            N_high = n
            prev_prev_N = None
            prev_growth = None
        
        N_low = max(1, N_high // 2)
        
        while True:
            print(f"    Testing N={N_high}...", end="", flush=True)
            success = 0
            for idx in range(num_samples):
                trace = ''.join(map(str, np.random.randint(0, q, n)))
                success += maximum_likelihood(trace, N_high, n, q, verbose=False)
                if (idx + 1) % 100 == 0:
                    print(".", end="", flush=True)
            
            success_rate = success / num_samples
            print(f" {success_rate:.1%} ({success}/{num_samples})")
            
            if success_rate >= target_success_rate:
                break
            
            N_low = N_high
            N_high = N_high + 10
        
        # PHASE 2: Binary search between N_low and N_high
        print(f"  Phase 2 - Binary search between N={N_low} and N={N_high}...")
        
        while N_high - N_low > 1:
            N_mid = (N_low + N_high) // 2
            print(f"    Testing N={N_mid}...", end="", flush=True)
            
            success = 0
            for idx in range(num_samples):
                trace = ''.join(map(str, np.random.randint(0, q, n)))
                success += maximum_likelihood(trace, N_mid, n, q, verbose=False)
                if (idx + 1) % 100 == 0:
                    print(".", end="", flush=True)
            
            success_rate = success / num_samples
            print(f" {success_rate:.1%} ({success}/{num_samples})")
            
            if success_rate >= target_success_rate:
                N_high = N_mid
            else:
                N_low = N_mid
        
        min_N = N_high
        print(f"           ✓ Found minimum N={min_N}")
        results[n] = min_N
        
        # Store for next iteration
        prev_N = min_N
        prev_n = n
    
    print("\n" + "=" * 75)
    print("SUMMARY - Vanilla Approach Results:")
    print("-" * 75)
    for n in n_values:
        print(f"  n={n:2d}  →  N={results[n]:>5d}")
    
    # Save results to CSV file
    import csv
    filename = 'sweep_N_vs_n_ml.csv'
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'N_min'])
        for n in n_values:
            writer.writerow([n, results[n]])
    print(f"✓ Results saved to '{filename}'")
    
    return results










In [ ]:
sweep_results_ml = sweep_N_vs_n_ml(
    target_success_rate=0.99,
    q=2,
    num_samples=1000
)

Sweeping N vs n using VANILLA maximum_likelihood
Target success rate: 0.99
Samples per test: 1000

n=10: (q=2, checking all 2^10=1024 candidates per test)
  Phase 1 - Finding upper bound...
    Testing N=10............. 57.1% (571/1000)
    Testing N=20............. 73.2% (732/1000)
    Testing N=30............. 83.6% (836/1000)
    Testing N=40............. 87.9% (879/1000)
    Testing N=50............. 89.9% (899/1000)
    Testing N=60............. 91.9% (919/1000)
    Testing N=70............. 91.7% (917/1000)
    Testing N=80............. 95.0% (950/1000)
    Testing N=90............. 97.0% (970/1000)
    Testing N=100............. 96.6% (966/1000)
    Testing N=110............. 97.4% (974/1000)
    Testing N=120............. 97.2% (972/1000)
    Testing N=130............. 98.3% (983/1000)
    Testing N=140............. 99.0% (990/1000)
  Phase 2 - Binary search between N=130 and N=140...
    Testing N=135............. 98.9% (989/1000)
    Testing N=137............. 98.1% (981/1000

In [ ]:
def sweep_N_vs_n_vanilla(target_success_rate=0.99, q=2, num_samples=1000):
    """
    Sweep N vs n using VANILLA maximum_likelihood (brute force approach).
    Uses exponential search + binary search to find EXACT minimum N.
    OPTIMIZED: Uses previous n results to estimate starting N for next n.
    n varies from 10 to 20 in intervals of 2.
    Tests with 1000 samples for robust success rate estimation.
    WARNING: This is computationally expensive but will give definitive results!
    """
    results = {}
    n_values = list(range(10, 21, 2)) 
    
    print(f"Sweeping N vs n using VANILLA maximum_likelihood")
    print(f"Target success rate: {target_success_rate}")
    print(f"Samples per test: {num_samples}")
    print("=" * 75)
    
    prev_N = None
    prev_n = None
    prev_growth = None
    
    for n in n_values:
        print(f"\nn={n}: (q={q}, checking all {q}^{n}={q**n} candidates per test)")
        
        # PHASE 1: Exponential search to find upper bound
        print(f"  Phase 1 - Finding upper bound...")
        
        # Smart initialization: use previous results to estimate starting point
        if prev_N is not None and prev_n is not None:
            delta_n = n - prev_n
            # Use observed growth rate (account for convergence)
            if prev_growth is not None:
                # Use moving average of growth to smooth out noise
                estimated_growth = (prev_growth + (prev_N / prev_prev_N if prev_prev_N else 1.0)) / 2
            else:
                # First step: estimate conservatively
                estimated_growth = (prev_N / prev_prev_N) if prev_prev_N else 1.2
            estimated_N = int(prev_N * (estimated_growth ** delta_n))
            N_high = max(estimated_N, prev_N)  # Don't go below previous N (should be monotonic)
            prev_growth = estimated_N / prev_N
            prev_prev_N = prev_N
        else:
            # First iteration: start with reasonable estimate
            N_high = n
            prev_prev_N = None
            prev_growth = None
        
        N_low = max(1, N_high // 2)
        
        while True:
            print(f"    Testing N={N_high}...", end="", flush=True)
            success = 0
            for idx in range(num_samples):
                trace = ''.join(map(str, np.random.randint(0, q, n)))
                success += maximum_likelihood(trace, N_high, n, q, verbose=False)
                if (idx + 1) % 100 == 0:
                    print(".", end="", flush=True)
            
            success_rate = success / num_samples
            print(f" {success_rate:.1%} ({success}/{num_samples})")
            
            if success_rate >= target_success_rate:
                break
            
            N_low = N_high
            N_high = int(N_high * 2)
        
        # PHASE 2: Binary search between N_low and N_high
        print(f"  Phase 2 - Binary search between N={N_low} and N={N_high}...")
        
        while N_high - N_low > 1:
            N_mid = (N_low + N_high) // 2
            print(f"    Testing N={N_mid}...", end="", flush=True)
            
            success = 0
            for idx in range(num_samples):
                trace = ''.join(map(str, np.random.randint(0, q, n)))
                success += maximum_likelihood(trace, N_mid, n, q, verbose=False)
                if (idx + 1) % 100 == 0:
                    print(".", end="", flush=True)
            
            success_rate = success / num_samples
            print(f" {success_rate:.1%} ({success}/{num_samples})")
            
            if success_rate >= target_success_rate:
                N_high = N_mid
            else:
                N_low = N_mid
        
        min_N = N_high
        print(f"           ✓ Found minimum N={min_N}")
        results[n] = min_N
        
        # Store for next iteration
        prev_N = min_N
        prev_n = n
    
    print("\n" + "=" * 75)
    print("SUMMARY - Vanilla Approach Results:")
    print("-" * 75)
    for n in n_values:
        print(f"  n={n:2d}  →  N={results[n]:>5d}")
    
    # Save results to CSV file
    import csv
    filename = 'sweep_N_vs_n_vanilla.csv'
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n', 'N_min'])
        for n in n_values:
            writer.writerow([n, results[n]])
    print(f"✓ Results saved to '{filename}'")
    
    return results

# Run the vanilla sweep
print("Starting OPTIMIZED VANILLA sweep with exponential+binary search...")
print("(1000 samples per test, smart starting points - Will find exact minimum N for each n)\n")
sweep_results_vanilla = sweep_N_vs_n_vanilla(target_success_rate=0.99, q=2, num_samples=1000)





Starting OPTIMIZED VANILLA sweep with exponential+binary search...
(1000 samples per test, smart starting points - Will find exact minimum N for each n)

Sweeping N vs n using VANILLA maximum_likelihood
Target success rate: 0.99
Samples per test: 1000

n=10: (q=2, checking all 2^10=1024 candidates per test)
  Phase 1 - Finding upper bound...
    Testing N=5............. 39.3% (393/1000)
    Testing N=10............. 53.9% (539/1000)
    Testing N=20............. 72.1% (721/1000)
    Testing N=40............. 85.4% (854/1000)
    Testing N=80............. 94.2% (942/1000)
    Testing N=160............. 98.7% (987/1000)
    Testing N=320............. 99.9% (999/1000)
  Phase 2 - Binary search between N=160 and N=320...
    Testing N=240............. 99.6% (996/1000)
    Testing N=200............. 99.7% (997/1000)
    Testing N=180............. 99.0% (990/1000)
    Testing N=170............. 98.9% (989/1000)
    Testing N=175............. 99.1% (991/1000)
    Testing N=172............. 99

In [ ]:

def ml_fast(trace, N, n, q=2, verbose=False):
    """
    Fast ML estimation with optimized exhaustive search through all q^n candidates.
    Core algorithm: Find candidate with maximum sum of common prefix lengths.
    Optimizations: Vectorized prefix computation, early termination, minimal allocations.
    """
    traces = channel1(trace, N, n, q)
    
    # Precompute traces as numpy arrays once
    traces_arr = np.array([np.array([int(c) for c in t], dtype=np.uint8) for t in traces])
    max_trace_len = traces_arr.shape[1] if traces_arr.size > 0 else 0
    
    def compute_likelihood_fast(candidate_arr):
        """Fast vectorized prefix length computation."""
        if len(candidate_arr) == 0:
            return 0
        
        cand_len = min(len(candidate_arr), max_trace_len)
        cand_prefix = candidate_arr[:cand_len]
        
        total_prefix = 0
        for trace_arr in traces_arr:
            # Find common prefix length
            if cand_len > 0:
                matches = (cand_prefix == trace_arr[:cand_len])
                if matches[0]:
                    # Find first mismatch or end
                    prefix_len = np.argmax(~matches) if np.any(~matches) else cand_len
                else:
                    prefix_len = 0
                total_prefix += prefix_len
        
        return total_prefix
    
    max_likelihood_score = -1
    best_candidate = None
    total_candidates = q ** n
    
    if verbose:
        print(f"Searching {total_candidates} candidates (q={q}, n={n})")
        start_time = time.time()
    
    # Exhaustive search (core algorithm unchanged)
    for idx, digits in enumerate(product(range(q), repeat=n)):
        if verbose and (idx % max(1, total_candidates // 10) == 0):
            elapsed = time.time() - start_time
            pct = 100 * idx / total_candidates
            print(f"  {idx:,}/{total_candidates:,} ({pct:.1f}%) - {elapsed:.1f}s")
        
        candidate_arr = np.array(digits, dtype=np.uint8)
        likelihood = compute_likelihood_fast(candidate_arr)
        
        if likelihood > max_likelihood_score:
            max_likelihood_score = likelihood
            best_candidate = ''.join(map(str, digits))
    
    success = 1 if best_candidate == trace else 0
    
    if verbose:
        elapsed = time.time() - start_time
        print(f"✓ Complete in {elapsed:.2f}s: {best_candidate} vs {trace} = {success}")
    
    return success

print("✓ ml_fast loaded")


In [11]:
success_rate_ml(1000,10,2)

0
Current success rate: 1.0000 after 1 samples
1
Current success rate: 1.0000 after 2 samples
2
Current success rate: 1.0000 after 3 samples
3
Current success rate: 1.0000 after 4 samples
4
Current success rate: 1.0000 after 5 samples
5
Current success rate: 1.0000 after 6 samples
6
Current success rate: 1.0000 after 7 samples
7
Current success rate: 1.0000 after 8 samples
8
Current success rate: 1.0000 after 9 samples
9
Current success rate: 1.0000 after 10 samples
10
Current success rate: 1.0000 after 11 samples
11
Current success rate: 1.0000 after 12 samples
12
Current success rate: 1.0000 after 13 samples
13
Current success rate: 1.0000 after 14 samples
14
Current success rate: 1.0000 after 15 samples
15
Current success rate: 1.0000 after 16 samples
16
Current success rate: 1.0000 after 17 samples
17
Current success rate: 1.0000 after 18 samples
18
Current success rate: 1.0000 after 19 samples
19
Current success rate: 1.0000 after 20 samples
20
Current success rate: 1.0000 after 2

KeyboardInterrupt: 

In [14]:
# Visualization of N vs n sweep results
import matplotlib.pyplot as plt

print("Creating visualization...")

if 'sweep_results' not in globals():
    print("Error: sweep_results not found. Run the sweep cell first.")
elif not sweep_results:
    print("Error: sweep_results is empty. The sweep may have failed.")
else:
    # Extract only successful results
    n_vals = sorted([n for n in sweep_results.keys() if sweep_results[n] is not None])
    N_vals = [sweep_results[n] for n in n_vals]
    
    if not n_vals:
        print("Error: No successful results found in sweep.")
    else:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Plot 1: Minimum N vs n (linear)
        ax1.plot(n_vals, N_vals, 'o-', linewidth=2.5, markersize=10, color='#2E86AB', label='Min N for 99%')
        ax1.set_xlabel('Sequence Length (n)', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Minimum Number of Traces (N)', fontsize=12, fontweight='bold')
        ax1.set_title('Required Traces vs Sequence Length', fontsize=13, fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--')
        ax1.set_xticks(n_vals)
        
        # Add value labels on points
        for n, N in zip(n_vals, N_vals):
            ax1.annotate(f'N={N}', xy=(n, N), xytext=(0, 12), 
                        textcoords='offset points', ha='center', fontsize=10, fontweight='bold')
        
        # Plot 2: Log scale to see exponential relationship
        ax2.semilogy(n_vals, N_vals, 's-', linewidth=2.5, markersize=10, color='#A23B72', label='Min N for 99%')
        ax2.set_xlabel('Sequence Length (n)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Minimum N (log scale)', fontsize=12, fontweight='bold')
        ax2.set_title('Required Traces (Logarithmic Scale)', fontsize=13, fontweight='bold')
        ax2.grid(True, alpha=0.3, which='both', linestyle='--')
        ax2.set_xticks(n_vals)
        
        # Add value labels on points
        for n, N in zip(n_vals, N_vals):
            ax2.annotate(f'{N}', xy=(n, N), xytext=(0, 10), 
                        textcoords='offset points', ha='center', fontsize=9)
        
        plt.tight_layout()
        plt.savefig('sweep_results.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("\n" + "="*70)
        print("ANALYSIS:")
        print("="*70)
        if len(n_vals) > 1:
            # Calculate growth rate
            growth_rate = (N_vals[-1] / N_vals[0]) ** (1 / (n_vals[-1] - n_vals[0]))
            print(f"Data points: {len(n_vals)}")
            print(f"N range: {min(N_vals)} - {max(N_vals)}")
            print(f"Exponential growth rate: ~{growth_rate:.2f}x per unit increase in n")
        print("="*70)
        print("✓ Visualization complete!")


Creating visualization...
Error: sweep_results not found. Run the sweep cell first.


In [15]:

# Visualization of ML sweep N vs n results
import matplotlib.pyplot as plt

print("Creating visualization for ML sweep results...")

if 'sweep_results_ml' not in globals():
    print("Error: sweep_results_ml not found. Run the sweep cell first.")
elif not sweep_results_ml:
    print("Error: sweep_results_ml is empty. The sweep may have failed.")
else:
    # Extract results
    n_vals_ml = sorted([n for n in sweep_results_ml.keys()])
    N_vals_ml = [sweep_results_ml[n] for n in n_vals_ml]
    
    if not n_vals_ml:
        print("Error: No results found in ML sweep.")
    else:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Plot 1: Minimum N vs n (linear scale)
        ax1.plot(n_vals_ml, N_vals_ml, 'o-', linewidth=2.5, markersize=12, 
                color='#1f77b4', label='Min N for 99% (ML)')
        ax1.set_xlabel('Sequence Length (n)', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Minimum Number of Traces (N)', fontsize=12, fontweight='bold')
        ax1.set_title('Required Traces vs Sequence Length (Maximum Likelihood)', fontsize=13, fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--')
        ax1.set_xticks(n_vals_ml)
        
        # Add value labels on points
        for n, N in zip(n_vals_ml, N_vals_ml):
            ax1.annotate(f'N={N}', xy=(n, N), xytext=(0, 12), 
                        textcoords='offset points', ha='center', fontsize=10, fontweight='bold')
        
        # Plot 2: Log scale to see exponential growth
        ax2.semilogy(n_vals_ml, N_vals_ml, 's-', linewidth=2.5, markersize=12, 
                    color='#ff7f0e', label='Min N for 99% (ML Log Scale)')
        ax2.set_xlabel('Sequence Length (n)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Minimum N (log scale)', fontsize=12, fontweight='bold')
        ax2.set_title('Required Traces (Logarithmic Scale - ML)', fontsize=13, fontweight='bold')
        ax2.grid(True, alpha=0.3, which='both', linestyle='--')
        ax2.set_xticks(n_vals_ml)
        
        # Add value labels
        for n, N in zip(n_vals_ml, N_vals_ml):
            ax2.annotate(f'{N}', xy=(n, N), xytext=(0, 10), 
                        textcoords='offset points', ha='center', fontsize=10)
        
        plt.tight_layout()
        plt.savefig('ml_sweep_results.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("\n" + "="*85)
        print("ML SWEEP ANALYSIS:")
        print("="*85)
        print(f"Data points: {len(n_vals_ml)}")
        print(f"n values: {n_vals_ml}")
        print(f"N values: {N_vals_ml}")
        print(f"Min N: {min(N_vals_ml)}")
        print(f"Max N: {max(N_vals_ml)}")
        
        if len(n_vals_ml) > 1:
            # Calculate growth rate
            growth_rate = (N_vals_ml[-1] / N_vals_ml[0]) ** (1 / (n_vals_ml[-1] - n_vals_ml[0]))
            print(f"Exponential growth rate: ~{growth_rate:.3f}x per unit increase in n")
            
            # Calculate ratio
            total_growth = N_vals_ml[-1] / N_vals_ml[0]
            n_diff = n_vals_ml[-1] - n_vals_ml[0]
            print(f"N grows {total_growth:.1f}x over Δn={n_diff}")
        
        print("="*85)
        print("✓ Visualization complete! Saved as 'ml_sweep_results.png'")


Creating visualization for ML sweep results...
Error: sweep_results_ml not found. Run the sweep cell first.
